# 19 — Full Simulation Run

The first **production-scale** run combining every feature added through v0.4. NB 17 was a 5×2 smoke test; NB 18 validated checkpoint/resume; this notebook scales those up to a publishable run and uses the new checkpoint machinery so a mid-run failure costs you minutes, not the whole run.

**Configuration (production defaults, edit at top of Section 4):**
- `N_CITIZENS = 50`, `N_DAYS = 7`
- Package communication mode (one broadcast / one peer pass per phase covers all 6 policies)
- `day0_anchor = "ground_truth_with_rationale"` — Day 0 mean equals the YouGov mean exactly; Day 1+ drift is unambiguously simulation-driven
- `debias = True` — Condition B two-step survey on every end-of-day call
- Dual-model: `gpt-5-mini` for messaging, `claude-sonnet-4-6` for surveys (extended thinking off by default)
- `checkpoint_every_day = True` — atomic per-day snapshot in `data/output/experiments/19_full_simulation_run/`. If the kernel dies mid-run, set `RESUME = True` in Section 4 and re-execute from Section 5 onwards.

**Wall-time / cost expectation.** NB 17 took ~30 min and ~£1.20 for 5×2. Naively scaling: 50×7 ≈ 35× the LLM calls. Budget **a few hours of wall-time** and check `timings.csv` after the run. Reduce `N_CITIZENS` or `N_DAYS` first if cost is a concern; everything else here works at any scale.

**Outputs.** All results land in a single timestamped directory: opinion / package-index trajectories, reflections, messages, survey reasoning, ground truth, share frames, plot PNGs, `config.json`, and `timings.csv`.


In [ ]:
import os, sys, random, logging, time, shutil
from contextlib import contextmanager
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import (
    ALL_CLIMATE_POLICIES,
    ClimatePolicyID,
    PACKAGE_SCOPE,
)
from cag.abm.sim import (
    run_simulation,
    save_results,
    save_result_plots,
    collect_ground_truth,
    collect_package_ground_truth,
    _load_checkpoint_meta,
)

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Timing harness ──────────────────────────────────────────────
TIMINGS = []

@contextmanager
def timed(label):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        TIMINGS.append((label, dt))
        print(f"[TIMER] {label}: {dt:.2f}s")

print("Imports OK")


## 1. Run Configuration

All knobs in one cell. Edit here to scale up/down or to flip on resume.


In [ ]:
# ── Headline knobs ──────────────────────────────────────────────
N_CITIZENS = 30
N_DAYS = 7
RANDOM_SEED = 43
year = 2026

# Network polarisation (default SBM): p_intra/p_inter ratio sets echo-chamber strength.
P_INTRA, P_INTER = 0.15, 0.02

# Resume control. Flip to True after a kernel crash to continue from the last
# completed-day checkpoint. The CHECKPOINT_DIR must already exist with a
# checkpoint_meta.json. RESUME=False with an existing dir wipes it (Section 3).
RESUME = True

# Output / checkpoint location. Keep it deterministic so resume can find it.
CHECKPOINT_DIR = Path("../data/output/experiments/19_full_simulation_run")

config = {
    "n_citizens": N_CITIZENS,
    "communication_mode": "package",
    "package_policies": list(ALL_CLIMATE_POLICIES),
    "day0_anchor": "ground_truth_with_rationale",
    "days": [
        {"phases": ["P-A", "P-B", "C"]},  # Day 1: pro-climate first
        {"phases": ["P-B", "P-A", "C"]},  # Day 2: anti-climate first
        {"phases": ["P-A", "P-B", "C"]},  # Day 3
        {"phases": ["P-B", "P-A", "C"]},  # Day 4
        {"phases": ["P-A", "P-B", "C"]},  # Day 5
        {"phases": ["P-B", "P-A", "C"]},  # Day 6
        {"phases": ["P-A", "P-B", "C"]},  # Day 7
    ],
    "k_peers_per_day": 3,
    "llm_model": "gpt-5-mini",
    "llm_provider": "openai",
    "llm_temperature": 0.5,
    "survey_model": "claude-sonnet-4-6",
    "survey_provider": "anthropic",
    "thinking": False,
    "debias": True,
    "p_intra": P_INTRA,
    "p_inter": P_INTER,
    "random_seed": RANDOM_SEED,
}

random.seed(RANDOM_SEED)

print(f"N_CITIZENS={N_CITIZENS}, N_DAYS={N_DAYS}, seed={RANDOM_SEED}")
print(f"Messaging: {config['llm_model']} ({config['llm_provider']})")
print(f"Surveys:   {config['survey_model']} ({config['survey_provider']}) "
      f"thinking={config['thinking']} debias={config['debias']}")
print(f"Day 0 anchor: {config['day0_anchor']}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"RESUME = {RESUME}")


## 2. Build Sample Nation

`build_nation()` is wrapped in a function so resume can call it again with the same seed and produce the identical agent set the checkpoint expects.


In [ ]:
UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)


def build_nation():
    sn = SurveyedNation(
        year=year, place="UK",
        gender_map=GenderMap(),
        region_map=UKRegionMap(),
        education_map=SurveyEducationMap(),
        ethnicity_map=SurveyEthnicityMap(),
        income_map=SurveyIncomeMap(),
        politics_map=SurveyPoliticsMap(),
        family_map=SurveyFamilyMap(),
        ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
        brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
        selftransc_map=SelftranscMap, selfenh_map=SelfenhMap,
        openness_map=OpennessMap, conformtrad_map=ConformTradMap,
        sdo_map=SDOMap, edo_map=EDOMap, rwa_map=RWAMap,
    )
    data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
    data = data.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)
    for i in range(len(data)):
        row = data.iloc[i]
        sc = SurveyedCitizen(
            agent_id=row.get('ID', None), environment=sn,
            year_of_birth=year - int(row.get('age', 0)),
            gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
            region_id=RegionID(int(row.get('tprofile_GOR', 0))),
            education_id=EducationID(int(row.get('profile_education_level', 0))),
            income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
            ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
            family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
            ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
            brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
            politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
            selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
            selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
            openness_id=rescale_1_6(int(row.get('Openness', 0))),
            conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
            sdo_id=rescale_1_7(int(row.get('SDO', 0))),
            edo_id=rescale_1_7(int(row.get('EDO', 0))),
            rwa_id=rescale_1_7(int(row.get('RWA', 0))),
            original_survey_data=data.iloc[i],
        )
        sn.agents_active[sc.id] = sc
    return sn


with timed("build_nation"):
    sn = build_nation()
print(f"Citizens loaded: {len(sn.agents_active)}")


## 3. Network Density + Ground Truth

Diagnostic only — `run_simulation()` will rebuild the network internally with the same seed. Mean degree should be comfortably above zero at N=50; if it's not, peer messaging will silently no-op for isolated agents.

Ground truth is also captured here so Section 7 can verify Day-0 anchoring exactly.


In [ ]:
sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")

with timed("create_network (diagnostic)"):
    sn.assign_political_exposure()
    sn.create_network(p_intra=P_INTRA, p_inter=P_INTER, seed=RANDOM_SEED)
    sn.assign_network_blocks()

degrees = [len(c.network_neighbors) for c in sn.agents_active.values()]
print(f"Mean degree: {np.mean(degrees):.2f}")
print(f"Agents with 0 neighbours: {sum(1 for d in degrees if d == 0)} / {len(degrees)}")
print(f"Total edges: {sn.network.number_of_edges()}")

with timed("collect_ground_truth"):
    gt_df = collect_ground_truth(sn.agents_active.values())
    gt_package_df = collect_package_ground_truth(sn.agents_active.values())

print(f"\nGT package mean: {gt_package_df['ground_truth'].mean():+.2f}  (n={len(gt_package_df)})")
print("GT mean per policy:")
for pid, grp in gt_df.groupby("policy_id"):
    print(f"  {str(pid).split('.')[-1]:25s} mean={grp['ground_truth'].mean():+.2f}")


## 4. Run Simulation (with per-day checkpoints)

This is the long cell. With `checkpoint_every_day=True` an atomic snapshot lands in `CHECKPOINT_DIR` after every day; if the kernel dies you can flip `RESUME = True` in Section 1, re-execute everything, and skip back to here to continue from the last completed day.

`KeyboardInterrupt` is caught so a manual stop still leaves a usable partial result on disk and a meaningful error in the notebook.


In [ ]:
# Resume vs fresh-start hygiene.
if RESUME:
    if not (CHECKPOINT_DIR / "checkpoint_meta.json").exists():
        raise FileNotFoundError(
            f"RESUME=True but no checkpoint at {CHECKPOINT_DIR}. "
            "Either flip RESUME=False to start fresh, or restore a checkpoint dir."
        )
    meta = _load_checkpoint_meta(CHECKPOINT_DIR)
    print(f"Resuming from checkpoint: last_completed_day={meta['last_completed_day']}, "
          f"written_at={meta['written_at']}")
    # Rebuild a clean nation; _load_checkpoint will hydrate it.
    sn = build_nation()
    sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
    sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
else:
    # Wipe any stale checkpoint so we don't accidentally hydrate from it.
    if CHECKPOINT_DIR.exists():
        print(f"Removing existing checkpoint dir: {CHECKPOINT_DIR}")
        shutil.rmtree(CHECKPOINT_DIR)

results = None
try:
    with timed("run_simulation (TOTAL)"):
        results = run_simulation(
            config, sn,
            checkpoint_dir=CHECKPOINT_DIR,
            resume=RESUME,
            checkpoint_every_day=True,
        )
    print(f"\nRun completed. Checkpoint dir: {CHECKPOINT_DIR}")
except KeyboardInterrupt:
    print(f"\n*** Interrupted. Partial checkpoint at: {CHECKPOINT_DIR} ***")
    print("    Set RESUME = True in Section 1 and re-execute to continue.")
    raise


## 5. Result Frames at a Glance


In [ ]:
df_traj = results["opinion_trajectories"]
df_package = results["package_index_trajectories"]
df_ref = results["reflections"]
df_reasoning = results["survey_reasoning"]
df_messages = results["messages"]

print(f"opinion_trajectories      : {len(df_traj):>6} rows  days={sorted(df_traj['day'].unique().tolist())}")
print(f"package_index_trajectories: {len(df_package):>6} rows")
print(f"reflections               : {len(df_ref):>6} rows")
print(f"survey_reasoning          : {len(df_reasoning):>6} rows")
print(f"messages                  : {len(df_messages):>6} rows")


## 6. Day-0 Anchor Verification

Sanity check that the GT-anchor actually fired: every (agent, policy) Day-0 opinion equals its YouGov ground truth, and every agent has a stored Day-0 rationale per policy.


In [ ]:
day0 = df_traj[df_traj["day"] == 0].copy()
day0["policy_id_str"] = day0["policy_id"].astype(str)
gt = gt_df.copy()
gt["policy_id_str"] = gt["policy_id"].astype(str)
merged = day0.merge(gt, on=["agent_id", "policy_id_str"], how="inner")
mismatches = merged[merged["numeric"] != merged["ground_truth"]]
assert mismatches.empty, f"Day 0 anchor mismatch on {len(mismatches)} rows:\n{mismatches.head()}"
print(f"PASS: Day 0 opinion == GT for all {len(merged)} (agent, policy) pairs")

day0_reasoning = df_reasoning[df_reasoning["day"] == 0]
expected = N_CITIZENS * len(ALL_CLIMATE_POLICIES)
print(f"Day 0 rationales stored: {len(day0_reasoning)} / expected {expected}")
assert len(day0_reasoning) == expected, "Missing Day 0 rationales"


## 7. Package Index Trajectory vs Ground Truth

Day 0 sits exactly on the GT mean (anchored). Day 1+ shows whether peer messaging and political broadcasts pull the population away.


In [ ]:
print("Mean package index by day:")
for day, val in df_package.groupby("day")["package_index"].mean().items():
    print(f"  Day {int(day)}: {val:+.2f}")
gt_pkg_mean = gt_package_df["ground_truth"].mean()
print(f"GT package mean: {gt_pkg_mean:+.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
for aid, sub in df_package.groupby("agent_id"):
    sub = sub.sort_values("day")
    ax.plot(sub["day"], sub["package_index"], alpha=0.25, color="steelblue",
            linewidth=0.8, marker="o", markersize=2.5)
llm_mean = df_package.groupby("day")["package_index"].mean()
ax.plot(llm_mean.index, llm_mean.values, color="black", linewidth=2.4,
        marker="o", label="LLM mean")
ax.axhline(gt_pkg_mean, color="red", linestyle="--", linewidth=1.5,
           label=f"GT package mean ({gt_pkg_mean:+.2f})")
ax.set_xlabel("Day")
ax.set_ylabel("Package index (−3 to +3)")
ax.set_ylim(-3.5, 3.5)
ax.set_title(f"Pro-Climate Package Index — N={N_CITIZENS}, {N_DAYS} days")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 8. Per-Policy Trajectories

One panel per policy: per-agent lines (light blue), LLM mean (black), GT mean (red dashed).


In [ ]:
POLICY_SHORT_NAMES = {
    str(ClimatePolicyID.RENEWABLE_ENERGY): "Renewable Energy",
    str(ClimatePolicyID.BAN_FOSSIL_FUEL): "Ban Fossil Fuels",
    str(ClimatePolicyID.BAN_PETROL_CARS): "Ban Petrol Cars",
    str(ClimatePolicyID.GREEN_HOUSING): "Green Housing",
    str(ClimatePolicyID.CARBON_TAX): "Carbon Tax",
    str(ClimatePolicyID.CLIMATE_COMPENSATION): "Climate Compensation",
}

policies_in_sim = sorted(df_traj["policy_id"].unique())
ncols = 3
nrows = (len(policies_in_sim) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for i, pid in enumerate(policies_in_sim):
    ax = axes[i // ncols, i % ncols]
    pdf = df_traj[df_traj["policy_id"] == pid]
    for aid, sub in pdf.groupby("agent_id"):
        sub = sub.sort_values("day")
        ax.plot(sub["day"], sub["numeric"], alpha=0.2, color="steelblue",
                linewidth=0.7, marker="o", markersize=2)
    llm_mean = pdf.groupby("day")["numeric"].mean()
    ax.plot(llm_mean.index, llm_mean.values, color="black", linewidth=2,
            marker="o", label="LLM mean")
    gt_p = gt_df[gt_df["policy_id"].astype(str) == str(pid)]
    if not gt_p.empty:
        gt_mean = gt_p["ground_truth"].mean()
        ax.axhline(gt_mean, color="red", linestyle="--", linewidth=1.3,
                   label=f"GT mean ({gt_mean:+.1f})")
    ax.set_xlabel("Day")
    ax.set_ylabel("Opinion")
    ax.set_ylim(-3.5, 3.5)
    ax.set_title(POLICY_SHORT_NAMES.get(str(pid), str(pid))[:40])
    ax.legend(fontsize=8)

for j in range(len(policies_in_sim), nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

fig.suptitle(f"Per-Policy Trajectories — N={N_CITIZENS}, {N_DAYS} days, Day 0 anchored", fontsize=13)
plt.tight_layout()
plt.show()


## 9. Activity Counts + Sample Outputs

(a) Reflection counts per phase, message-log size.
(b) One sampled reflection per phase.
(c) One Day≥1 debias survey reasoning entry.


In [ ]:
print("Reflections per phase:")
print(df_ref.groupby("phase").size().to_string())
print(f"\nMessages logged: {len(df_messages)}")

print("\n--- Sample reflection per phase ---")
for phase, grp in df_ref.groupby("phase"):
    if grp.empty:
        continue
    row = grp.iloc[0]
    text = row.get("reflection") or row.get("text") or ""
    print(f"\n[{phase}] agent={row['agent_id']} day={row['day']}")
    print(text[:400])

print("\n--- Sample Day>=1 debias reasoning ---")
late = df_reasoning[df_reasoning["day"] > 0]
if late.empty:
    print("(no Day>=1 reasoning rows)")
else:
    row = late.iloc[0]
    print(f"agent={row['agent_id']} day={row['day']} policy={row['policy_id']}")
    print(row["reasoning"][:500])


## 10. Save Final Results + Plots + Timings

Persists the canonical end-of-run outputs into a fresh timestamped directory under `data/output/experiments/` (separate from the in-progress checkpoint dir, which can be deleted after this cell succeeds). Also writes `timings.csv` so we know where wall-time went.


In [ ]:
out_path = save_results(results, output_dir="../data/output/experiments")
plot_paths = save_result_plots(results, out_path)

timings_df = pd.DataFrame(TIMINGS, columns=["section", "seconds"])
total = timings_df["seconds"].sum()
timings_df.to_csv(out_path / "timings.csv", index=False)

print(f"\nResults: {out_path}")
print(f"Plots:   {len(plot_paths)} files")
print(f"\n=== Wall-time summary ({total:.1f}s = {total/60:.1f} min) ===")
print(timings_df.to_string(index=False))
print(f"\nTimings written to {out_path / 'timings.csv'}")
print(f"\nCheckpoint dir ({CHECKPOINT_DIR}) can now be deleted if desired.")
